# CCLE lung cancer: concordance across RNA, RPPA, and metabolomics

This is a matched-cohort follow-up to the global and within-tissue trees. It
builds independent Ward trees for each assay, evaluates the agreement of their
branch assignments, and makes a feature-stacked Clustergram with an explicit
`modality` row annotation.

In [1]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import adjusted_rand_score


PROJECT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from celldega.viz import Clustergram
from notebooks.ccle_multiomics import (
    HierarchicalConfig,
    hierarchical_clustergram,
    load_collection,
    marker_clustergram,
    rank_cluster_markers,
    shared_model_ids,
    stack_modalities,
)


DATA = PROJECT / "data/ccle_multiomics"
ARCHIVE = DATA / "processed/ccle_multiomics.h5mu"
RESULTS = DATA / "results/04_lung_multiomic_concordance"
RESULTS.mkdir(parents=True, exist_ok=True)

MODALITIES = ("rna", "rppa", "metabolomics")
N_BRANCHES = 3
FEATURES_PER_MODALITY = {"rna": 2_000, "rppa": 214, "metabolomics": 225}
# A stacked Euclidean tree should not be dominated simply because RNA has more
# assayed features. Keep independent trees source-appropriate above, then use
# the same number of high-variance rows from each assay in the integrated view.
STACKED_FEATURES_PER_MODALITY = {"rna": 214, "rppa": 214, "metabolomics": 214}

collection = load_collection(ARCHIVE)
shared = shared_model_ids(collection, MODALITIES)
rna = collection.mod["rna"]
lung_models = shared[rna.obs.loc[shared, "OncotreeLineage"].eq("Lung")]
print(f"Matched lung cohort: {len(lung_models)} models")

Matched lung cohort: 157 models


## Independent per-assay trees

The branches below are discovered independently. `marker_clustergrams` holds
top-marker slider views for interactive inspection, while the tables retain
rank statistics and source-scale median effects.

In [2]:
independent: dict[str, object] = {}
marker_clustergrams: dict[str, object] = {}
label_tables: list[pd.DataFrame] = []
marker_tables: list[pd.DataFrame] = []

for modality in MODALITIES:
    analysis, _ = hierarchical_clustergram(
        collection.mod[modality][lung_models].copy(),
        HierarchicalConfig(
            n_top_features=FEATURES_PER_MODALITY[modality],
            n_meta_clusters=N_BRANCHES,
        ),
    )
    cluster_key = analysis.uns["hierarchical_clustergram"]["cluster_key"]
    independent[modality] = analysis

    labels = analysis.obs[[
        field
        for field in ("OncotreePrimaryDisease", "OncotreeSubtype", cluster_key)
        if field in analysis.obs
    ]].copy()
    labels.insert(0, "ModelID", analysis.obs_names)
    labels.insert(1, "modality", modality)
    labels = labels.rename(columns={cluster_key: "hierarchical_cluster"})
    label_tables.append(labels)

    markers = rank_cluster_markers(analysis, cluster_key, key_added="rank_genes_groups")
    marker_clustergrams[modality] = marker_clustergram(analysis)
    markers.insert(0, "modality", modality)
    marker_tables.append(markers)

independent_labels = pd.concat(label_tables, ignore_index=True)
independent_markers = pd.concat(marker_tables, ignore_index=True)
independent_labels.to_parquet(RESULTS / "matched_lung_cluster_labels.parquet", index=False)
independent_markers.to_parquet(RESULTS / "matched_lung_hierarchical_markers.parquet", index=False)
independent_markers.query("rank <= 20 and pvals_adj <= 0.05").to_csv(
    RESULTS / "matched_lung_top20_markers.csv", index=False
)

## Co-clustering coincidence on the same cell lines

The adjusted Rand index compares branch membership without requiring an
arbitrary relabeling. The contingency tables show which RNA branches coincide
with an RPPA or metabolite branch on this explicit matched cohort.

In [3]:
labels_by_modality = {
    modality: independent[modality].obs["hierarchical_k3"].astype(str) for modality in MODALITIES
}
agreement_rows: list[dict[str, object]] = []

for left, right in combinations(MODALITIES, 2):
    table = pd.crosstab(labels_by_modality[left], labels_by_modality[right])
    table.index.name = f"{left}_cluster"
    table.columns.name = f"{right}_cluster"
    table.to_csv(RESULTS / f"{left}_by_{right}_branch_coincidence.csv")
    agreement_rows.append(
        {
            "left_modality": left,
            "right_modality": right,
            "n_models": len(lung_models),
            "adjusted_rand_index": adjusted_rand_score(
                labels_by_modality[left], labels_by_modality[right]
            ),
        }
    )

agreement = pd.DataFrame(agreement_rows)
agreement.to_csv(RESULTS / "independent_branch_agreement.csv", index=False)
print(agreement)

  left_modality right_modality  n_models  adjusted_rand_index
0           rna           rppa       157             0.357632
1           rna   metabolomics       157             0.122574
2          rppa   metabolomics       157             0.099756


## A stacked multi-omic Clustergram

`stack_modalities` retains the matched cohort and prefixes feature IDs only to
keep them unique. Its `modality` row annotation distinguishes RNA genes, RPPA
antibodies, and metabolites while the row z-score gives their patterns a
common display scale. Equal feature counts prevent RNA from winning simply
because it has a much larger measured feature space.

In [4]:
stacked = stack_modalities(
    collection,
    MODALITIES,
    lung_models,
    n_top_features=STACKED_FEATURES_PER_MODALITY,
)
stacked_analysis, stacked_clustergram = hierarchical_clustergram(
    stacked,
    HierarchicalConfig(n_top_features=stacked.n_vars, n_meta_clusters=N_BRANCHES),
)
stacked_key = stacked_analysis.uns["hierarchical_clustergram"]["cluster_key"]
stacked_markers = rank_cluster_markers(stacked_analysis, stacked_key, key_added="rank_genes_groups")
stacked_marker_clustergram = marker_clustergram(stacked_analysis)

stacked_labels = stacked_analysis.obs[[
    field
    for field in ("OncotreePrimaryDisease", "OncotreeSubtype", stacked_key)
    if field in stacked_analysis.obs
]].copy()
stacked_labels.insert(0, "ModelID", stacked_analysis.obs_names)
stacked_labels = stacked_labels.rename(columns={stacked_key: "hierarchical_cluster"})
stacked_labels.to_parquet(RESULTS / "stacked_lung_cluster_labels.parquet", index=False)
stacked_markers.to_parquet(RESULTS / "stacked_lung_hierarchical_markers.parquet", index=False)

stacked_agreement = []
for modality, labels in labels_by_modality.items():
    stacked_agreement.append(
        {
            "modality": modality,
            "n_models": len(lung_models),
            "adjusted_rand_index_with_stacked_tree": adjusted_rand_score(
                labels, stacked_analysis.obs[stacked_key].astype(str)
            ),
        }
    )
pd.DataFrame(stacked_agreement).to_csv(RESULTS / "stacked_branch_agreement.csv", index=False)

## Interactive lung stacked Clustergram

This view places the matched lung lines on one shared dendrogram. Its
`modality` track distinguishes RNA genes, RPPA antibodies, and metabolites;
the RANK control shows top features for each of the three dendrogram branches.

In [5]:
Clustergram(matrix=stacked_marker_clustergram, width=1_100, height=700)

Clustergram(category_colors={'Lung': '#56D3D8', 'Lung Neuroendocrine Tumor': '#8156D8', 'Non-Small Cell Lung C…